5350 Thesis in Economics

Andres Cruz 25199 & Klara Holmer 25037 

Code for Skolenkäten  

In [40]:
# Step 0: Importing Libraries
import os
import glob
import pandas as pd

In [41]:
# Step 1: Skolenkäten - Gymnasieskola

# Define the folder path
path = r'/Users/andrescruz/Documents/Handelshögskolan/MSc Economic/Semester 4/5350 Thesis in Economics/Raw Data/Skolenkäten Data/gymnasieskola'

# Get and sort Excel files
all_files = glob.glob(os.path.join(path, "*.xlsx"))
all_files.sort()

# Initialize list to store row-wise results
data_rows = []

# Process each file
for file in all_files:
    try:
        df = pd.read_excel(file, header=None, sheet_name=0)

        # Locate the row with "Samtliga deltagande skolor"
        row_idx = df[df.apply(lambda row: row.astype(str).str.contains("Samtliga deltagande skolor", na=False).any(), axis=1)].index
        row_pos = row_idx[0]

        # Find columns that match "Index" or "Indexvärde"
        index_cols = [
            col_idx for col_idx in df.columns
            if df[col_idx].astype(str).str.contains(r"\bIndex\b|\bIndexvärde\b", case=False, na=False).any()
        ]

        # Collect values in one pass
        row_data = {
            'file_name': os.path.basename(file),
            'educational_stage': 'gymnasieskola'
        }

        for q_num, col in enumerate(index_cols, start=1):
            row_data[f'Q{q_num}'] = df.iat[row_pos, col]

        data_rows.append(row_data)
        print(f'Processed: {os.path.basename(file)}')

    except Exception as e:
        print(f"Error processing file: {os.path.basename(file)}")
        print(f"Exception: {e}")
        continue

# Assemble wide-format DataFrame
gymnasieskola_df = pd.DataFrame(data_rows)

Processed: skolenkäten_2015_1_vt.xlsx
Processed: skolenkäten_2015_2_ht.xlsx
Processed: skolenkäten_2016_1_vt.xlsx
Processed: skolenkäten_2016_2_ht.xlsx
Processed: skolenkäten_2017_1_vt.xlsx
Processed: skolenkäten_2017_2_ht.xlsx
Processed: skolenkäten_2018_1_vt.xlsx
Processed: skolenkäten_2018_2_ht.xlsx
Processed: skolenkäten_2019_1_vt.xlsx
Processed: skolenkäten_2019_2_ht.xlsx
Processed: skolenkäten_2020_1_vt.xlsx


In [42]:
# Step 2: Skolenkäten - Grundskola

# Define the folder path for grundskola
path = r'/Users/andrescruz/Documents/Handelshögskolan/MSc Economic/Semester 4/5350 Thesis in Economics/Raw Data/Skolenkäten Data/grundskola'

# Get and sort Excel files
all_files = glob.glob(os.path.join(path, "*.xlsx"))
all_files.sort()

# Initialize list to store row-wise results
data_rows = []

# Process each file
for file in all_files:
    try:
        df = pd.read_excel(file, header=None, sheet_name=0)

        # Locate the row with "Samtliga deltagande skolor"
        row_idx = df[df.apply(lambda row: row.astype(str).str.contains("Samtliga deltagande skolor", na=False).any(), axis=1)].index
        row_pos = row_idx[0]

        # Find columns that match "Index" or "Indexvärde"
        index_cols = [
            col_idx for col_idx in df.columns
            if df[col_idx].astype(str).str.contains(r"\bIndex\b|\bIndexvärde\b", case=False, na=False).any()
        ]
        
        # Collect values in one pass
        row_data = {
            'file_name': os.path.basename(file),
            'educational_stage': 'grundskola'
        }

        for q_num, col in enumerate(index_cols, start=1):
            row_data[f'Q{q_num}'] = df.iat[row_pos, col]

        data_rows.append(row_data)
        print(f'Processed: {os.path.basename(file)}')

    except Exception as e:
        print(f"Error processing file: {os.path.basename(file)}")
        print(f"Exception: {e}")
        continue

# Assemble wide-format DataFrame
grundskola_df = pd.DataFrame(data_rows)

Processed: skolenkäten_2015_1_vt.xlsx
Processed: skolenkäten_2015_2_ht.xlsx
Processed: skolenkäten_2016_1_vt.xlsx
Processed: skolenkäten_2016_2_ht.xlsx
Processed: skolenkäten_2017_1_vt.xlsx
Processed: skolenkäten_2017_2_ht.xlsx
Processed: skolenkäten_2018_1_vt.xlsx
Processed: skolenkäten_2018_2_ht.xlsx
Processed: skolenkäten_2019_1_vt.xlsx
Processed: skolenkäten_2019_2_ht.xlsx
Processed: skolenkäten_2020_1_vt.xlsx


In [ ]:
# Step 3: Concatinating, Modifying, and Exporting

# Concatinating 
skolenkaten = pd.concat([gymnasieskola_df, grundskola_df], ignore_index=True)

# Extracting year and semester from file names, and combining them into Year Semester 
skolenkaten[['year', 'semester']] = skolenkaten['file_name'].str.extract(r'(\d{4}).*?(vt|ht)', expand=True) # Extracting the correct strings
skolenkaten['semester'] = skolenkaten['semester'].map({'vt': 'Spring', 'ht': 'Autumn'}) # Map Swedish semester codes to English
skolenkaten['year_semester'] = skolenkaten['year'] + ' ' + skolenkaten['semester'] # Combine into a single year_semester column

# Fixing formating of the Question values
question_cols = [f'Q{i}' for i in range(1, 16)] # Ensure Q1 to Q15 exist and are numeric with six decimals
skolenkaten[question_cols] = skolenkaten[question_cols].apply(pd.to_numeric, errors='coerce').round(6) # Convert to numeric and round

# Defining the full path where the Excel file will be saved
pathname = '/Users/andrescruz/Documents/Handelshögskolan/MSc Economic/Semester 4/5350 Thesis in Economics/Processed Data/skolenkäten.xlsx'

# Exporting all the tables created into one Excel File 
with pd.ExcelWriter(pathname) as writer:
    skolenkaten.to_excel(writer, sheet_name = 'Skolenkäten', index=False)